In [3]:
import os
import numpy as np
import pandas as pd


df_crops = pd.read_csv("D:/vscode/NASA/PRE DATA/crops_database.csv")

print("=" * 75)
print(f"🌱 إجمالي المحاصيل الموثقة: {df_crops.shape[0]} محصول × {df_crops.shape[1]} خاصية")
print(f"🧹 القيم المفقودة: {df_crops.isnull().sum().sum()}")
print("=" * 75)

print("\n📊 توزيع المحاصيل حسب العائلة النباتية (Crop_Family) والموسم (Season):")
display(pd.crosstab(df_crops["Crop_Family"], df_crops["Season"]))

print("\n👀 عينة من المحاصيل (أول 10 محاصيل):")
display(df_crops[["Crop_ID", "Crop_Name_AR", "Crop_Name_EN", "Crop_Family", "Season", "Crop_Water_Requirement_ETc_mm", "Gross_Nitrogen_Fixation_kg_ha", "Drought_Tolerance", "Heat_Tolerance", "Economic_Market_Value"]].head(10))

🌱 إجمالي المحاصيل الموثقة: 51 محصول × 18 خاصية
🧹 القيم المفقودة: 0

📊 توزيع المحاصيل حسب العائلة النباتية (Crop_Family) والموسم (Season):


Season,Inter-season,Perennial,Summer,Winter
Crop_Family,,,,
Amaranthaceae,0,0,0,2
Amaryllidaceae,0,0,0,2
Apiaceae,0,0,0,3
Araceae,0,0,1,0
Asteraceae,0,1,1,0
Brassicaceae,1,0,0,2
Convolvulaceae,0,0,1,0
Cucurbitaceae,0,0,4,0
Fabaceae,0,1,4,7



👀 عينة من المحاصيل (أول 10 محاصيل):


,Crop_ID,Crop_Name_AR,Crop_Name_EN,Crop_Family,Season,Crop_Water_Requirement_ETc_mm,Gross_Nitrogen_Fixation_kg_ha,Drought_Tolerance,Heat_Tolerance,Economic_Market_Value
0,C01,قمح,Wheat,Poaceae,Winter,450,0,Medium,Medium,Very High
1,C02,ذرة شامية بيضاء,White Maize,Poaceae,Summer,650,0,Low,Medium,High
2,C03,ذرة صفراء,Yellow Maize,Poaceae,Summer,620,0,Low,Medium,Very High
3,C04,شعير,Barley,Poaceae,Winter,300,0,High,High,Medium
4,C05,ذرة رفيعة (حبوب),Grain Sorghum,Poaceae,Summer,450,0,Very High,Very High,Medium
5,C06,أرز,Rice,Poaceae,Summer,1100,0,Low,Medium,Very High
6,C07,شوفان,Oats,Poaceae,Winter,380,0,Medium,Low,Medium
7,C08,تريتيكال,Triticale,Poaceae,Winter,390,0,High,Medium,Medium
8,C09,دخن,Pearl Millet,Poaceae,Summer,300,0,Very High,Very High,Low
9,L01,برسيم مسقاوي,Egyptian Clover (Berseem),Fabaceae,Winter,550,180,Medium,Medium,High


In [4]:
# عرض جميع المحاصيل غير التابعة للعائلة النجيلية (Poaceae) لاختيار أفضل 3 دورات بديلة
non_poaceae = df_crops[df_crops["Crop_Family"] != "Poaceae"][[
    "Crop_ID", "Crop_Name_AR", "Crop_Name_EN", "Crop_Family", "Season",
    "Crop_Water_Requirement_ETc_mm", "Gross_Nitrogen_Fixation_kg_ha",
    "Drought_Tolerance", "Heat_Tolerance", "Soil_Salinity_Threshold_ECe_dS_m", "Economic_Market_Value"
]]
display(non_poaceae)

,Crop_ID,Crop_Name_AR,Crop_Name_EN,Crop_Family,Season,Crop_Water_Requirement_ETc_mm,Gross_Nitrogen_Fixation_kg_ha,Drought_Tolerance,Heat_Tolerance,Soil_Salinity_Threshold_ECe_dS_m,Economic_Market_Value
9,L01,برسيم مسقاوي,Egyptian Clover (Berseem),Fabaceae,Winter,550,180,Medium,Medium,3.2,High
10,L02,فول بلدي,Faba Bean,Fabaceae,Winter,380,120,Medium,Low,1.6,Very High
11,L03,فول صويا,Soybean,Fabaceae,Summer,500,100,Medium,Medium,5.0,Very High
12,L04,حمص,Chickpea,Fabaceae,Winter,280,70,High,Medium,1.5,High
13,L05,عدس,Lentil,Fabaceae,Winter,250,60,High,Low,2.0,Very High
14,L06,لوبيا صيفية,Cowpea,Fabaceae,Summer,350,90,Very High,Very High,4.9,Medium
15,L07,برسيم حجازي,Alfalfa,Fabaceae,Perennial,1200,250,High,High,2.0,High
16,L08,ترمس,Lupine,Fabaceae,Winter,320,110,High,Medium,2.5,High
17,L09,بسلة خضراء,Green Peas,Fabaceae,Winter,300,50,Low,Low,1.5,High
18,L10,فاصوليا جافة,Common Dry Bean,Fabaceae,Summer,360,45,Low,Medium,1.0,Very High


In [5]:
def evaluate_crop_rotations(
    df_crops: pd.DataFrame,
    governorate: str,
    prev_crop_ar: str,
    pred_sm: float,
    uncertainty: float,
    pred_tmax: float,
    soil_ph: float,
    soil_clay: float,
    w_water: float = 0.40,
    w_soil: float = 0.35,
    w_econ: float = 0.25,
) -> pd.DataFrame:
    """
    محرك القرار متعدد المعايير (MCDA) لمقارنة 4 استراتيجيات للدورة الزراعية في مصر
    بناءً على تنبؤات أقمار ناسا وقاعدة بيانات المحاصيل الموثقة (51 محصولاً).
    """
    total_w = max(w_water + w_soil + w_econ, 1e-6)
    w_water, w_soil, w_econ = w_water / total_w, w_soil / total_w, w_econ / total_w

    prev_row = df_crops[df_crops["Crop_Name_AR"] == prev_crop_ar]
    prev_family = prev_row["Crop_Family"].values[0] if len(prev_row) > 0 else "Unknown"

    crops_idx = df_crops.set_index("Crop_ID")

    strategies = [
        {
            "id": "S1_SOIL_RESTORE",
            "name": "استراتيجية استعادة خصوبة التربة وكسر الآفات",
            "type": "بديل ذكي (Smart Rotation)",
            "seq_ids": ["L02", "L06", "O02"],  # فول بلدي -> لوبيا صيفية -> بنجر السكر
            "why": "تُدخل محصولين بقوليين (فول بلدي ولوبيا) لتثبيت 210 كجم/هكتار من النيتروجين الجوي طبيعياً، متبوعين ببنجر السكر ذي الجذور العميقة لتفكيك التربة وتحقيق عائد نقدي مرتفع.",
            "benefits": "توفير 31% من مياه الري، خفض الحاجة للأسمدة الأزوتية الكيماوية بنسبة 35%، وكسر دورة حشائش وأمراض الحبوب النجيلية.",
            "risks": "يتطلب الفول البلدي مقاومة مبكرة لحشيشة الهالوك؛ كما يُنصح بمراجعة المرشد الزراعي لضبط مواعيد التسميد البوتاسي لبنجر السكر."
        },
        {
            "id": "S2_DROUGHT_HEAT_SHIELD",
            "name": "استراتيجية المقاومة القصوى للجفاف وموجات الحر",
            "type": "بديل ذكي (Smart Rotation)",
            "seq_ids": ["C04", "O05", "L04"],  # شعير -> سمسم -> حمص
            "why": "مصممة خصيصاً لمواجهة ندرة المياه وارتفاع الحرارة العظمى (+35°C)، حيث تجمع بين محاصيل فائقة الكفاءة المائية (شعير، سمسم، حمص) من 3 عائلات نباتية مختلفة تماماً.",
            "benefits": "توفير قياسي يصل إلى 55% من إجمالي مياه الري مقارنة بالدورة التقليدية، وتقليل خسائر الإجهاد الحراري، وتنويع كامل للعائلات النباتية.",
            "risks": "السمسم والحمص حساسان لتغدق التربة وسوء الصرف في الأراضي الطينية الثقيلة؛ يلزم تسوية الأرض بالليزر واستشارة خبير محلي لتنظيم فترات الري."
        },
        {
            "id": "S3_SALINITY_OIL_PROFIT",
            "name": "استراتيجية التوازن الربحي والزيتي وتحمل الملوحة",
            "type": "بديل ذكي (Smart Rotation)",
            "seq_ids": ["O06", "L03", "C01"],  # كانولا -> فول صويا -> قمح
            "why": "تدمج الكانولا (عالية التحمل للملوحة حتى 11 dS/m) مع فول الصويا المثبت للنيتروجين والمنتج للزيت والبروتين، لتهيئة تربة خصبة ونظيفة لمحصول القمح الاستراتيجي.",
            "benefits": "توفير 32.5% من المياه، سد فجوة الزيوت النباتية، تحقيق قيمة سوقية مرتفعة جداً، وحماية المحصول في الأراضي المتأثرة بالملوحة.",
            "risks": "تحتاج الكانولا إلى عقود تسويق مسبقة مع مصانع استخلاص الزيوت وتحديد كثافة البذور المناسبة بالتنسيق مع الإرشاد الزراعي."
        },
        {
            "id": "S0_TRADITIONAL",
            "name": "الدورة التقليدية السائدة (خط الأساس للمقارنة)",
            "type": "دورة تقليدية (Baseline)",
            "seq_ids": ["C01", "C06", "C01"],  # قمح -> أرز -> قمح
            "why": "تمثل النمط الزراعي التقليدي المعتمد على تكرار الحبوب النجيلية (Poaceae) لتحقيق الاكتفاء الذاتي المباشر من الحبوب.",
            "benefits": "توفر قنوات تسويق تقليدية مستقرة وخبرة زراعية متوارثة لدى المزارعين في عمليات الخدمة والحصاد.",
            "risks": "استنزاف حاد لمياه الري (2000 مم/دورة)، صفر تثبيت حيوي للنيتروجين، وإجهاد التربة مع تراكم آفات النجيليات."
        },
    ]

    tol_map = {"Low": 0.25, "Medium": 0.55, "High": 0.80, "Very High": 1.0}
    om_map = {"Low": 0.25, "Medium": 0.50, "High": 0.80, "Very High": 1.0}
    econ_map = {"Low": 0.25, "Medium": 0.50, "High": 0.75, "Very High": 1.0}

    # حساب مؤشرات الإجهاد المناخي من تنبؤات موديل ناسا
    drought_stress = float(np.clip((0.30 - pred_sm) / 0.15, 0.0, 1.0))
    heat_stress = float(np.clip((pred_tmax - 32.0) / 10.0, 0.0, 1.0))
    climate_stress = 0.6 * drought_stress + 0.4 * heat_stress

    # خط الأساس التقليدي للمقارنة المرجعية للمياه
    base_sub = crops_idx.loc[["C01", "C06", "C01"]]
    baseline_water_total = float(base_sub["Crop_Water_Requirement_ETc_mm"].sum())  # 2000 mm

    rows = []
    for s in strategies:
        sub = crops_idx.loc[s["seq_ids"]]

        seq_ar = " ⬅️ ".join(sub["Crop_Name_AR"].tolist())
        seq_en = " -> ".join(sub["Crop_Name_EN"].tolist())
        families = sub["Crop_Family"].tolist()

        total_etc = float(sub["Crop_Water_Requirement_ETc_mm"].sum())
        water_saved_mm = round(baseline_water_total - total_etc, 1)
        water_saved_pct = round((water_saved_mm / baseline_water_total) * 100, 1)
        total_n_fix = float(sub["Gross_Nitrogen_Fixation_kg_ha"].sum())
        avg_salinity_tol = round(float(sub["Soil_Salinity_Threshold_ECe_dS_m"].mean()), 1)

        resilience = float(
            0.5 * sub["Drought_Tolerance"].map(tol_map).mean()
            + 0.5 * sub["Heat_Tolerance"].map(tol_map).mean()
        )

        # 1. درجة كفاءة المياه (0-100) مع معايرة للنطاق الفعلي (850 - 2100 mm)
        water_score = float(np.clip((2150.0 - total_etc) / 1300.0 * 100.0, 0.0, 100.0))

        # 2. درجة صحة وخصوبة التربة (تثبيت النيتروجين + المادة العضوية + تنوع العائلات)
        unique_families_ratio = len(set(families)) / len(families)
        soil_score = float(np.clip(
            (min(total_n_fix, 220.0) / 220.0 * 45.0)
            + (sub["Organic_Matter_Contribution"].map(om_map).mean() * 35.0)
            + (unique_families_ratio * 20.0),
            0.0, 100.0
        ))

        # 3. درجة العائد الاقتصادي (0-100)
        econ_score = float(sub["Economic_Market_Value"].map(econ_map).mean() * 100.0)

        # 4. العقوبات الفيزيائية والزراعية (الإجهاد المناخي + تكرار عائلة المحصول السابق + قلوية التربة)
        first_family = families[0]
        prev_crop_penalty = 12.0 if first_family == prev_family else 0.0
        climate_penalty = climate_stress * (1.0 - resilience) * 35.0
        ph_penalty = 4.0 if (soil_ph > 8.1 and avg_salinity_tol < 3.0) else 0.0

        raw_score = (
            (w_water * water_score)
            + (w_soil * soil_score)
            + (w_econ * econ_score)
            - climate_penalty
            - prev_crop_penalty
            - ph_penalty
        )
        final_score = round(float(np.clip(raw_score, 0.0, 100.0)), 1)

        nasa_evidence = (
            f"تُظهر بيانات رصد الأرض لمحافظة {governorate} رطوبة تربة متوقعة {pred_sm:.3f} ± {uncertainty:.3f} m³/m³ "
            f"وحرارة عظمى {pred_tmax:.1f}°C مع قلوية تربة pH={soil_ph:.1f} ونسبة طين {soil_clay:.0f}%."
        )
        data_gaps = (
            "الأقمار الصناعية تقيس الرطوبة السطحية والغطاء النباتي بدقة إقليمية، لكنها لا تقيس ملوحة التربة الحقلية (ECe) "
            "أو ملوحة مياه الآبار أو توفر التقاوي محلياً؛ لذا يُنصح بمراجعة المهندس الزراعي قبل التنفيذ."
        )

        rows.append({
            "Strategy_ID": s["id"],
            "Strategy_Name": s["name"],
            "Strategy_Type": s["type"],
            "Rotation_Sequence_AR": seq_ar,
            "Rotation_Sequence_EN": seq_en,
            "Final_Score": final_score,
            "Water_Score": round(water_score, 1),
            "Soil_Score": round(soil_score, 1),
            "Econ_Score": round(econ_score, 1),
            "Total_Water_ETc_mm": round(total_etc, 0),
            "Water_Saved_mm": water_saved_mm,
            "Water_Saved_Pct": water_saved_pct,
            "Nitrogen_Fixed_kg_ha": round(total_n_fix, 0),
            "Avg_Salinity_ECe_dS_m": avg_salinity_tol,
            "Pest_Cycle_Status": "⚠️ تكرار عائلة المحصول السابق" if prev_crop_penalty > 0 else "✅ تتابع سليم يكسر الآفات",
            "Why_Suggested": s["why"],
            "NASA_Evidence": nasa_evidence,
            "Benefits": s["benefits"],
            "Local_Risks": s["risks"],
            "Data_Gaps_Disclaimer": data_gaps,
        })

    return pd.DataFrame(rows).sort_values("Final_Score", ascending=False).reset_index(drop=True)

print("✅ تم بناء محرك القرار متعدد المعايير (MCDA Engine) بنجاح!")

✅ تم بناء محرك القرار متعدد المعايير (MCDA Engine) بنجاح!


In [8]:
import joblib

# 1. تحميل حزمة موديل الذكاء الاصطناعي وبيانات المناخ
MODEL_PATH = os.path.join(r"d:\vscode\NASA\DATA", "models", "soil_gb_model.joblib")
CLIMATE_PATH = os.path.join(r"d:\vscode\NASA\DATA", "data", "Egypt_Model_Ready_Clean.csv")

artifact = joblib.load(MODEL_PATH)
df_clim = pd.read_csv(CLIMATE_PATH)
df_clim["Date"] = pd.to_datetime(df_clim["Date"])
df_clim = df_clim.sort_values(["Governorate", "Date"]).reset_index(drop=True)

# 2. إنشاء الخصائص الزمنية ومتغيرات التأخير (Lag/Rolling) المطلوبة للموديل
df_clim["Year"] = df_clim["Date"].dt.year
df_clim["Month"] = df_clim["Date"].dt.month
df_clim["DayOfYear"] = df_clim["Date"].dt.dayofyear

df_clim["Lag1_SoilMoi_Surface"] = df_clim.groupby("Governorate")["ERA5_SoilMoi_0_7cm_cm3"].shift(1)
df_clim["Lag1_SoilMoi_Root"] = df_clim.groupby("Governorate")["SMAP_RootZone_Moisture_cm3"].shift(1)
df_clim["Lag1_ET0"] = df_clim.groupby("Governorate")["ERA5_ET0_mm"].shift(1)
df_clim["Lag1_Water_Balance"] = df_clim.groupby("Governorate")["Net_Water_Balance_mm"].shift(1)
df_clim["Roll3_SoilMoi_Surface"] = (
    df_clim.groupby("Governorate")["ERA5_SoilMoi_0_7cm_cm3"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)
df_clim = df_clim.dropna().reset_index(drop=True)

# 3. محاكاة سيناريو مزارع في محافظة الشرقية (+1.5°C جفاف متوسط)
gov = "Sharqia"
temp_shift = 1.5
gov_sample = df_clim[df_clim["Governorate"] == gov].tail(6).copy()
gov_sample["POWER_Temp_Mean_C"] += temp_shift
gov_sample["POWER_Temp_Max_C"] += temp_shift
gov_sample["ERA5_ET0_mm"] *= (1.0 + 0.04 * temp_shift)
gov_sample["Net_Water_Balance_mm"] = gov_sample["POWER_Precip_mm_day"] - gov_sample["ERA5_ET0_mm"]

X_infer = gov_sample[artifact["features"]]
pred_sm = float(artifact["model_mean"].predict(X_infer).mean())
pred_low = float(artifact["model_lower"].predict(X_infer).mean())
pred_high = float(artifact["model_upper"].predict(X_infer).mean())
unc = (pred_high - pred_low) / 2.0

# 4. تشغيل محرك القرار ومقارنة الاستراتيجيات الأربع
df_recommendations = evaluate_crop_rotations(
    df_crops=df_crops,
    governorate=gov,
    prev_crop_ar="قمح",
    pred_sm=pred_sm,
    uncertainty=unc,
    pred_tmax=float(gov_sample["POWER_Temp_Max_C"].mean()),
    soil_ph=float(gov_sample["Soil_pH_Topsoil"].iloc[-1]),
    soil_clay=float(gov_sample["Soil_Clay_Pct"].iloc[-1]),
    w_water=0.45,
    w_soil=0.35,
    w_econ=0.20,
)

print(f"🛰️ توقع رطوبة التربة لمحافظة {gov} (سيناريو +{temp_shift}°C): {pred_sm:.3f} ± {unc:.3f} m³/m³\n")
display(df_recommendations[[
    "Strategy_Name", "Rotation_Sequence_AR", "Final_Score",
    "Water_Saved_Pct", "Nitrogen_Fixed_kg_ha", "Avg_Salinity_ECe_dS_m", "Pest_Cycle_Status"
]])

print("\n📋 التقرير التفصيلي لأفضل استراتيجية مُرشّحة:")
best = df_recommendations.iloc[0]
for k in ["Why_Suggested", "NASA_Evidence", "Benefits", "Local_Risks", "Data_Gaps_Disclaimer"]:
    print(f"- {k}: {best[k]}")

🛰️ توقع رطوبة التربة لمحافظة Sharqia (سيناريو +1.5°C): 0.130 ± 0.012 m³/m³



,Strategy_Name,Rotation_Sequence_AR,Final_Score,Water_Saved_Pct,Nitrogen_Fixed_kg_ha,Avg_Salinity_ECe_dS_m,Pest_Cycle_Status
0,استراتيجية استعادة خصوبة التربة وكسر الآفات,فول بلدي ⬅️ لوبيا صيفية ⬅️ بنجر السكر,64.2,31.0,210.0,4.5,✅ تتابع سليم يكسر الآفات
1,استراتيجية المقاومة القصوى للجفاف وموجات الحر,شعير ⬅️ سمسم ⬅️ حمص,59.7,55.0,70.0,4.2,⚠️ تكرار عائلة المحصول السابق
2,استراتيجية التوازن الربحي والزيتي وتحمل الملوحة,كانولا (شلجم) ⬅️ فول صويا ⬅️ قمح,58.3,32.5,100.0,7.3,✅ تتابع سليم يكسر الآفات
3,الدورة التقليدية السائدة (خط الأساس للمقارنة),قمح ⬅️ أرز ⬅️ قمح,10.1,0.0,0.0,5.0,⚠️ تكرار عائلة المحصول السابق



📋 التقرير التفصيلي لأفضل استراتيجية مُرشّحة:
- Why_Suggested: تُدخل محصولين بقوليين (فول بلدي ولوبيا) لتثبيت 210 كجم/هكتار من النيتروجين الجوي طبيعياً، متبوعين ببنجر السكر ذي الجذور العميقة لتفكيك التربة وتحقيق عائد نقدي مرتفع.
- NASA_Evidence: تُظهر بيانات رصد الأرض لمحافظة Sharqia رطوبة تربة متوقعة 0.130 ± 0.012 m³/m³ وحرارة عظمى 28.0°C مع قلوية تربة pH=8.1 ونسبة طين 26%.
- Benefits: توفير 31% من مياه الري، خفض الحاجة للأسمدة الأزوتية الكيماوية بنسبة 35%، وكسر دورة حشائش وأمراض الحبوب النجيلية.
- Local_Risks: يتطلب الفول البلدي مقاومة مبكرة لحشيشة الهالوك؛ كما يُنصح بمراجعة المرشد الزراعي لضبط مواعيد التسميد البوتاسي لبنجر السكر.
- Data_Gaps_Disclaimer: الأقمار الصناعية تقيس الرطوبة السطحية والغطاء النباتي بدقة إقليمية، لكنها لا تقيس ملوحة التربة الحقلية (ECe) أو ملوحة مياه الآبار أو توفر التقاوي محلياً؛ لذا يُنصح بمراجعة المهندس الزراعي قبل التنفيذ.


In [9]:
MCDA_MODULE_PATH = os.path.join(r"d:\vscode\NASA\DATA", "models", "mcda_engine.py")

mcda_code = '''import numpy as np
import pandas as pd


def evaluate_crop_rotations(
    df_crops: pd.DataFrame,
    governorate: str,
    prev_crop_ar: str,
    pred_sm: float,
    uncertainty: float,
    pred_tmax: float,
    soil_ph: float,
    soil_clay: float,
    w_water: float = 0.40,
    w_soil: float = 0.35,
    w_econ: float = 0.25,
) -> pd.DataFrame:
    total_w = max(w_water + w_soil + w_econ, 1e-6)
    w_water, w_soil, w_econ = w_water / total_w, w_soil / total_w, w_econ / total_w

    prev_row = df_crops[df_crops["Crop_Name_AR"] == prev_crop_ar]
    prev_family = prev_row["Crop_Family"].values[0] if len(prev_row) > 0 else "Unknown"

    crops_idx = df_crops.set_index("Crop_ID")

    strategies = [
        {
            "id": "S1_SOIL_RESTORE",
            "name": "استراتيجية استعادة خصوبة التربة وكسر الآفات",
            "type": "بديل ذكي (Smart Rotation)",
            "seq_ids": ["L02", "L06", "O02"],
            "why": "تُدخل محصولين بقوليين (فول بلدي ولوبيا) لتثبيت 210 كجم/هكتار من النيتروجين الجوي طبيعياً، متبوعين ببنجر السكر ذي الجذور العميقة لتفكيك التربة وتحقيق عائد نقدي مرتفع.",
            "benefits": "توفير 31% من مياه الري، خفض الحاجة للأسمدة الأزوتية الكيماوية بنسبة 35%، وكسر دورة حشائش وأمراض الحبوب النجيلية.",
            "risks": "يتطلب الفول البلدي مقاومة مبكرة لحشيشة الهالوك؛ كما يُنصح بمراجعة المرشد الزراعي لضبط مواعيد التسميد البوتاسي لبنجر السكر."
        },
        {
            "id": "S2_DROUGHT_HEAT_SHIELD",
            "name": "استراتيجية المقاومة القصوى للجفاف وموجات الحر",
            "type": "بديل ذكي (Smart Rotation)",
            "seq_ids": ["C04", "O05", "L04"],
            "why": "مصممة خصيصاً لمواجهة ندرة المياه وارتفاع الحرارة العظمى (+35°C)، حيث تجمع بين محاصيل فائقة الكفاءة المائية (شعير، سمسم، حمص) من 3 عائلات نباتية مختلفة تماماً.",
            "benefits": "تخفيض استهلاك مياه الري بأكثر من 55% مقارنة بالدورة التقليدية، وتقليل خسائر الإجهاد الحراري، وتنويع كامل للعائلات النباتية.",
            "risks": "السمسم والحمص حساسان لتغدق التربة وسوء الصرف في الأراضي الطينية الثقيلة؛ يلزم تسوية الأرض بالليزر واستشارة خبير محلي لتحديد فترات الري."
        },
        {
            "id": "S3_SALINITY_OIL_PROFIT",
            "name": "استراتيجية التوازن الربحي والزيتي وتحمل الملوحة",
            "type": "بديل ذكي (Smart Rotation)",
            "seq_ids": ["O06", "L03", "C01"],
            "why": "تدمج الكانولا (عالية التحمل للملوحة حتى 11 dS/m) مع فول الصويا المثبت للنيتروجين والمنتج للزيت والبروتين، لتهيئة تربة خصبة ونظيفة لمحصول القمح الاستراتيجي.",
            "benefits": "توفير 32.5% من المياه، سد فجوة الزيوت النباتية، تحقيق قيمة سوقية مرتفعة جداً، وحماية المحصول في الأراضي المتأثرة بالملوحة.",
            "risks": "تحتاج الكانولا إلى عقود تسويق مسبقة مع مصانع استخلاص الزيوت وتحديد كثافة البذور المناسبة بالتنسيق مع الإرشاد الزراعي."
        },
        {
            "id": "S0_TRADITIONAL",
            "name": "الدورة التقليدية السائدة (خط الأساس للمقارنة)",
            "type": "دورة تقليدية (Baseline)",
            "seq_ids": ["C01", "C06", "C01"],
            "why": "تمثل النمط الزراعي التقليدي المعتمد على تكرار الحبوب النجيلية (Poaceae) المتتالية لتحقيق الاكتفاء الذاتي المباشر من الحبوب.",
            "benefits": "توفر قنوات تسويق تقليدية مستقرة وخبرة زراعية متوارثة لدى المزارعين في عمليات الخدمة والحصاد.",
            "risks": "استنزاف حاد لمياه الري (2000 مم/دورة)، صفر تثبيت حيوي للنيتروجين، وإجهاد التربة مع تراكم آفات النجيليات."
        },
    ]

    tol_map = {"Low": 0.25, "Medium": 0.55, "High": 0.80, "Very High": 1.0}
    om_map = {"Low": 0.25, "Medium": 0.50, "High": 0.80, "Very High": 1.0}
    econ_map = {"Low": 0.25, "Medium": 0.50, "High": 0.75, "Very High": 1.0}

    drought_stress = float(np.clip((0.30 - pred_sm) / 0.15, 0.0, 1.0))
    heat_stress = float(np.clip((pred_tmax - 32.0) / 10.0, 0.0, 1.0))
    climate_stress = 0.6 * drought_stress + 0.4 * heat_stress

    base_sub = crops_idx.loc[["C01", "C06", "C01"]]
    baseline_water_total = float(base_sub["Crop_Water_Requirement_ETc_mm"].sum())

    rows = []
    for s in strategies:
        sub = crops_idx.loc[s["seq_ids"]]

        seq_ar = " ⬅️ ".join(sub["Crop_Name_AR"].tolist())
        seq_en = " -> ".join(sub["Crop_Name_EN"].tolist())
        families = sub["Crop_Family"].tolist()

        total_etc = float(sub["Crop_Water_Requirement_ETc_mm"].sum())
        water_saved_mm = round(baseline_water_total - total_etc, 1)
        water_saved_pct = round((water_saved_mm / baseline_water_total) * 100, 1)
        total_n_fix = float(sub["Gross_Nitrogen_Fixation_kg_ha"].sum())
        avg_salinity_tol = round(float(sub["Soil_Salinity_Threshold_ECe_dS_m"].mean()), 1)

        resilience = float(
            0.5 * sub["Drought_Tolerance"].map(tol_map).mean()
            + 0.5 * sub["Heat_Tolerance"].map(tol_map).mean()
        )

        water_score = float(np.clip((2150.0 - total_etc) / 1300.0 * 100.0, 0.0, 100.0))
        unique_families_ratio = len(set(families)) / len(families)
        soil_score = float(np.clip(
            (min(total_n_fix, 220.0) / 220.0 * 45.0)
            + (sub["Organic_Matter_Contribution"].map(om_map).mean() * 35.0)
            + (unique_families_ratio * 20.0),
            0.0, 100.0
        ))
        econ_score = float(sub["Economic_Market_Value"].map(econ_map).mean() * 100.0)

        first_family = families[0]
        prev_crop_penalty = 12.0 if first_family == prev_family else 0.0
        climate_penalty = climate_stress * (1.0 - resilience) * 35.0
        ph_penalty = 4.0 if (soil_ph > 8.1 and avg_salinity_tol < 3.0) else 0.0

        raw_score = (
            (w_water * water_score)
            + (w_soil * soil_score)
            + (w_econ * econ_score)
            - climate_penalty
            - prev_crop_penalty
            - ph_penalty
        )
        final_score = round(float(np.clip(raw_score, 0.0, 100.0)), 1)

        nasa_evidence = (
            f"تُظهر بيانات رصد الأرض لمحافظة {governorate} رطوبة تربة متوقعة {pred_sm:.3f} ± {uncertainty:.3f} m³/m³ "
            f"وحرارة عظمى {pred_tmax:.1f}°C مع قلوية تربة pH={soil_ph:.1f} ونسبة طين {soil_clay:.0f}%."
        )
        data_gaps = (
            "الأقمار الصناعية تقيس الرطوبة السطحية والغطاء النباتي بدقة إقليمية، لكنها لا تقيس ملوحة التربة الحقلية (ECe) "
            "أو ملوحة مياه الآبار أو توفر التقاوي محلياً؛ لذا يُنصح بمراجعة المهندس الزراعي قبل التنفيذ."
        )

        rows.append({
            "Strategy_ID": s["id"],
            "Strategy_Name": s["name"],
            "Strategy_Type": s["type"],
            "Rotation_Sequence_AR": seq_ar,
            "Rotation_Sequence_EN": seq_en,
            "Final_Score": final_score,
            "Water_Score": round(water_score, 1),
            "Soil_Score": round(soil_score, 1),
            "Econ_Score": round(econ_score, 1),
            "Total_Water_ETc_mm": round(total_etc, 0),
            "Water_Saved_mm": water_saved_mm,
            "Water_Saved_Pct": water_saved_pct,
            "Nitrogen_Fixed_kg_ha": round(total_n_fix, 0),
            "Avg_Salinity_ECe_dS_m": avg_salinity_tol,
            "Pest_Cycle_Status": "⚠️ تكرار عائلة المحصول السابق" if prev_crop_penalty > 0 else "✅ تتابع سليم يكسر الآفات",
            "Why_Suggested": s["why"],
            "NASA_Evidence": nasa_evidence,
            "Benefits": s["benefits"],
            "Local_Risks": s["risks"],
            "Data_Gaps_Disclaimer": data_gaps,
        })

    return pd.DataFrame(rows).sort_values("Final_Score", ascending=False).reset_index(drop=True)
'''

with open(MCDA_MODULE_PATH, "w", encoding="utf-8") as f:
    f.write(mcda_code)

print("=" * 75)
print("🎉 اكتملت جميع مهام العضو الثالث بنجاح تام!")
print(f"📦 تم تصدير محرك القرار الجاهز في: {MCDA_MODULE_PATH}")
print("=" * 75)

🎉 اكتملت جميع مهام العضو الثالث بنجاح تام!
📦 تم تصدير محرك القرار الجاهز في: d:\vscode\NASA\DATA\models\mcda_engine.py
